<a href="https://colab.research.google.com/github/eeeaaai/YL/blob/main/async_federated_learning_versiyon_3_noniid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:


import numpy as np
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Tue Feb 18 22:41:42 2025

@author: egemenalacali
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import threading
import time
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


# ----------------------------
# Device and Backend Optimizations for Colab Pro
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
torch.backends.cudnn.benchmark = True

# ----------------------------
# FedAsync Hyperparameters
# ----------------------------
NUM_CLIENTS = 3                   # Number of worker clients
MAX_GLOBAL_ITER = 5              # Total number of global iterations (T)
ALPHA = 0.1                       # Base update factor (α)
BATCH_SIZE = 64
NUM_LOCAL_EPOCHS = 1              # Local epochs per client update
LEARNING_RATE = 0.01              # Local learning rate
RHO = 0.01                        # Regularization coefficient (ρ)

# DataLoader workers (adjust based on runtime and GPU)
NUM_WORKERS = 2

# Global iteration counter (t)
global_iteration = 0
global_lock = threading.Lock()    # Protect global model updates

# ----------------------------
# Define a simple CNN for CIFAR-10 Classification
# ----------------------------
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # CIFAR-10 images are 3x32x32
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)   # Output: 32x32x32
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)    # Output: 64x32x32
        self.pool = nn.MaxPool2d(2, 2)                              # Halves spatial dims
        self.dropout = nn.Dropout(0.25)
        # After two poolings: 64 x 8 x 8 = 4096 features
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)                               # 10 classes for CIFAR-10

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)         # 32 x 16 x 16
        x = F.relu(self.conv2(x))
        x = self.pool(x)         # 64 x 8 x 8
        x = self.dropout(x)
        x = x.view(x.size(0), -1) # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ----------------------------
# Global Model Initialization (x0)
# ----------------------------
global_model = CNN().to(device)

# ----------------------------
# Staleness Function s(t - τ)
# ----------------------------
def staleness_decay(tau: int) -> float:
    """Returns a decay factor that discounts stale updates."""
    return 1.0 / (tau + 1)

# ----------------------------
# Updater Function (Server Side)
# Receives (x_new, τ) and integrates update:
#   x_{t+1} = (1 - α_t)x_t + α_t x_new, with α_t = α × s(t-τ)
# ----------------------------
def updater(client_model_state, client_timestamp):
    global global_model, global_iteration
    with global_lock:
        tau = global_iteration - client_timestamp  # staleness
        alpha_t = ALPHA * staleness_decay(tau)

        # Merge client's update into the global model
        global_state = global_model.state_dict()
        for key in global_state:
            global_state[key] = (1 - alpha_t) * global_state[key] + alpha_t * client_model_state[key]
        global_model.load_state_dict(global_state)

        global_iteration += 1
        print(f"[Updater] Global Iteration: {global_iteration}, tau: {tau}, α_t: {alpha_t:.4f}")

# ----------------------------
# Worker Process (Client Side)
# Each worker:
# 1. Receives (x_t, t) from the server.
# 2. Initializes its local model and defines the local loss:
#    g_{x_t}(x; z) = f(x; z) + (ρ/2)*||x - x_t||².
# 3. Performs local training for H iterations.
# 4. Sends (x_t^i, t) to the updater.
# ----------------------------
def process_worker(client_id, train_loader):
    global global_model, global_iteration
    criterion = nn.CrossEntropyLoss()
    local_updates_done = 0

    while True:
        with global_lock:
            if global_iteration >= MAX_GLOBAL_ITER:
                break
            # 1. Receive current global model (x_t) and timestamp t
            local_model = CNN().to(device)
            local_model.load_state_dict(global_model.state_dict())
            # Snapshot for regularization term (x_t)
            global_snapshot = CNN().to(device)
            global_snapshot.load_state_dict(global_model.state_dict())
            local_timestamp = global_iteration

        optimizer = optim.SGD(local_model.parameters(), lr=LEARNING_RATE)
        local_model.train()

        # 2. Local training for NUM_LOCAL_EPOCHS epochs
        for epoch in range(NUM_LOCAL_EPOCHS):
            for batch_idx, (data, target) in enumerate(train_loader):
                data, target = data.to(device), target.to(device)
                optimizer.zero_grad()
                output = local_model(data)
                loss = criterion(output, target)

                # Regularization: (ρ/2)*||x - x_t||^2, penalize deviation from global_snapshot
                reg_loss = 0.0
                for param, global_param in zip(local_model.parameters(), global_snapshot.parameters()):
                    reg_loss += torch.norm(param - global_param)**2
                loss += (RHO / 2.0) * reg_loss

                loss.backward()
                optimizer.step()

                if batch_idx % 100 == 0:
                    print(f"[Worker {client_id}] Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item():.4f}")

        print(f"[Worker {client_id}] Completed update #{local_updates_done}, Last Loss: {loss.item():.4f}")
        # 3. Send the updated model and the timestamp to the updater
        updater(local_model.state_dict(), local_timestamp)
        local_updates_done += 1

        # 4. Simulate asynchronous delay
        time.sleep(random.uniform(0.5, 2.0))

# ----------------------------
# Scheduler Function
# Spawns and manages worker threads.
# ----------------------------
def scheduler(client_loaders):
    threads = []
    for client_id in range(NUM_CLIENTS):
        t = threading.Thread(target=process_worker, args=(client_id, client_loaders[client_id]))
        t.start()
        threads.append(t)

    for t in threads:
        t.join()

# ----------------------------
# Global Evaluation on CIFAR-10 Test Set
# ----------------------------
def evaluate_global_model(test_loader):
    global global_model
    global_model.eval()  # Set model to evaluation mode
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = global_model(images)  # Forward pass
            _, preds = torch.max(outputs, 1)  # Get predicted class
            all_preds.extend(preds.cpu().numpy())  # Store predictions
            all_labels.extend(labels.cpu().numpy())  # Store actual labels

    # Compute the confusion matrix using scikit-learn
    cm = confusion_matrix(all_labels, all_preds)
    classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']  # CIFAR-10 classes

    # Convert counts to row-wise percentages
    row_sums = cm.sum(axis=1, keepdims=True)
    # Avoid division by zero if a row sum is 0
    cm_percent = np.divide(cm, row_sums, where=row_sums!=0) * 100
    cm_percent = np.round(cm_percent, 2)

    # Plot the percentage confusion matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_percent, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted Label (%)")
    plt.ylabel("True Label (%)")
    plt.title("Confusion Matrix (Percentage) for CIFAR-10")
    plt.show()



def main():
    # CIFAR-10 data transforms and normalization
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    # Load CIFAR-10 training and test datasets
    train_dataset = datasets.CIFAR10("./data", train=True, download=True, transform=transform)
    test_dataset = datasets.CIFAR10("./data", train=False, download=True, transform=transform)

    # ----------------------------
    # Non-IID Data Partitioning
    # ----------------------------
    train_size = len(train_dataset)  # Total number of training samples

    # Randomly assign different sizes to each client (non-uniform distribution)
    min_samples = train_size // (NUM_CLIENTS * 3)  # Minimum per client
    max_samples = train_size // NUM_CLIENTS  # Maximum per client
    client_sizes = np.random.randint(min_samples, max_samples, size=NUM_CLIENTS)

    # Adjust last client to ensure all samples are used
    client_sizes[-1] = train_size - sum(client_sizes[:-1])

    # Partition dataset based on assigned sizes
    client_subsets = random_split(train_dataset, client_sizes.tolist())

    # Create DataLoaders for each client with pin_memory and multiple workers
    client_loaders = [DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True,
                                 num_workers=NUM_WORKERS, pin_memory=True) for subset in client_subsets]

    # Create DataLoader for the test set
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

    print(f"Non-IID Client Data Distribution: {client_sizes}")

    # Run the scheduler to spawn worker threads
    scheduler(client_loaders)

    # Evaluate the final global model on the test set
    evaluate_global_model(test_loader)

if __name__ == "__main__":
    main()

Using device: cuda


100%|██████████| 170M/170M [00:04<00:00, 41.2MB/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified
Non-IID Client Data Distribution: [ 9009 15742 25249]
[Worker 2] Epoch 0, Batch 0, Loss: 2.3071[Worker 1] Epoch 0, Batch 0, Loss: 2.2945

[Worker 0] Epoch 0, Batch 0, Loss: 2.3050
[Worker 2] Epoch 0, Batch 100, Loss: 2.2801
[Worker 2] Epoch 0, Batch 200, Loss: 2.2245
[Worker 2] Epoch 0, Batch 300, Loss: 2.0691
[Worker 2] Completed update #0, Last Loss: 2.2251
[Updater] Global Iteration: 1, tau: 0, α_t: 0.1000
[Worker 2] Epoch 0, Batch 0, Loss: 2.2970
[Worker 2] Epoch 0, Batch 100, Loss: 2.2237
[Worker 2] Epoch 0, Batch 200, Loss: 2.1908
[Worker 2] Epoch 0, Batch 300, Loss: 1.9925
[Worker 2] Completed update #1, Last Loss: 1.8063
[Updater] Global Iteration: 2, tau: 0, α_t: 0.1000
[Worker 2] Epoch 0, Batch 0, Loss: 2.2591
[Worker 2] Epoch 0, Batch 100, Loss: 2.1292
[Worker 2] Epoch 0, Batch 200, Loss: 2.0872
[Worker 2] Epoch 0, Batch 300, Loss: 2.0048
[Worker 2] Completed update #2, Last Loss: 1.8678

KeyboardInterrupt: 